# Aggregate phase-1 dataset audit interface

This notebook defines the extensible dataset config interface. Future datasets should be added by adding one config entry here and one matching download notebook.

In [ ]:
from pathlib import Path

PHASE1_DATASETS = [
    {
        "dataset_id": "icbhi_2017",
        "raw_dir": Path("../raw/icbhi_2017"),
        "processed_dir": Path("../processed/icbhi_2017"),
        "label_sources": ["per-recording annotation txt", "events.zip"],
        "metadata_sources": ["filename", "demographic txt", "diagnosis txt", "train/test txt"],
        "unit_of_analysis": ["recording", "respiratory_cycle"],
        "raw_label_columns": ["crackle_flag", "wheeze_flag"],
        "patient_key_fields": ["patient_id"],
        "device_key_fields": ["recording_equipment"],
        "split_fields": ["official_train_test"],
        "normal_abnormal_mapping": {"normal": "crackle=0 and wheeze=0", "abnormal": "crackle=1 or wheeze=1"},
        "event_label_mapping": {"normal": "0,0", "crackle": "1,0", "wheeze": "0,1", "both": "1,1"},
        "supported_heads": ["normal_abnormal", "crackle_wheeze_both_normal", "dataset_specific_cycle"],
    },
    {
        "dataset_id": "hf_lung_v1",
        "raw_dir": Path("../raw/hf_lung_v1"),
        "processed_dir": Path("../processed/hf_lung_v1"),
        "label_sources": ["*_label.txt"],
        "metadata_sources": ["filename", "README.md"],
        "unit_of_analysis": ["15s_recording", "event_interval"],
        "raw_label_columns": ["inhalation", "exhalation", "CAS", "DAS"],
        "patient_key_fields": ["date/session proxy"],
        "device_key_fields": ["filename prefix steth_/trunc_"],
        "split_fields": ["source train/test folders"],
        "normal_abnormal_mapping": {"abnormal": "CAS or DAS interval", "normal": "unknown until background protocol reviewed"},
        "event_label_mapping": {"crackle": "DAS", "wheeze": "CAS subtype wheeze", "other_adventitious": "stridor/rhonchi"},
        "supported_heads": ["dataset_specific_event_detection", "normal_abnormal_likely_after_review"],
    },
    {
        "dataset_id": "kauh_fraiwan",
        "raw_dir": Path("../raw/kauh_fraiwan"),
        "processed_dir": Path("../processed/kauh_fraiwan"),
        "label_sources": ["Data annotation.xlsx"],
        "metadata_sources": ["filename", "Data annotation.xlsx", "Mendeley page"],
        "unit_of_analysis": ["recording/file"],
        "raw_label_columns": ["sound type", "diagnosis", "location"],
        "patient_key_fields": ["patient number P1-P112"],
        "device_key_fields": ["3M Littmann 3200"],
        "split_fields": [],
        "normal_abnormal_mapping": {"normal": "sound type Normal", "abnormal": "wheeze/crackle/other non-normal after workbook review"},
        "event_label_mapping": {"crackle": "C", "wheeze": "W", "normal": "N", "both": "unknown / needs audit"},
        "supported_heads": ["sound_type", "diagnosis_after_separation"],
    },
    {
        "dataset_id": "sprsound",
        "raw_dir": Path("../raw/sprsound"),
        "processed_dir": Path("../processed/sprsound"),
        "label_sources": ["JSON record_annotation", "JSON event_annotation"],
        "metadata_sources": ["filename", "Patient Summary CSV", "README.md"],
        "unit_of_analysis": ["recording", "event"],
        "raw_label_columns": ["record_annotation", "event_annotation.type"],
        "patient_key_fields": ["patient number"],
        "device_key_fields": ["Yunting model II Stethoscope"],
        "split_fields": ["challenge-year directory", "inter/intra test folders"],
        "normal_abnormal_mapping": {"normal": "Normal", "abnormal": "CAS/DAS/Rhonchi/Wheeze/Stridor/Crackle/Wheeze+Crackle", "exclude_or_separate": "Poor Quality"},
        "event_label_mapping": {"crackle": "Fine Crackle/Coarse Crackle", "wheeze": "Wheeze", "both": "Wheeze+Crackle", "other_adventitious": "Rhonchi/Stridor"},
        "supported_heads": ["record_level", "event_level", "normal_abnormal", "dataset_specific_multiclass"],
    },
]

PHASE1_DATASETS

## Build cross-dataset summary artifacts

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from phase1_dataset_workflow import main

main(["phase1_dataset_workflow.py", "summary"])

## Preview phase-1 matrix

In [ ]:
import pandas as pd

matrix = Path("../processed/phase1_dataset_matrix_2026-06-29.csv")
if matrix.exists():
    display(pd.read_csv(matrix))
else:
    print("Run the summary cell after individual dataset audits are complete.")